# Thomson-1.0-Small

Seven-arm experiment on [`thomsonreuters/Thomson-1.0-Small`](https://huggingface.co/thomsonreuters/Thomson-1.0-Small). Isolated from the main sweep on size: 70.2 GB weights, A100 80GB (High RAM).

Checkpoint card: base `tri-fair-lab/Snowdon1.1-Small`, architecture `Qwen3_5MoeForConditionalGeneration`. The model is a shipped legal-product weight, post-trained (Constitutional DPO / conformance RL); that may move deference relative to the open bases.


## Licence

Thomson-1.0-Small is **PolyForm Strict 1.0.0**. Permitted: research, experiment, and testing for public knowledge; use by educational and public research organisations; fair use. No restriction on benchmarking or publishing results. Forbidden: distributing the software or making derivative works — this notebook does neither.

The grant is for a noncommercial purpose. An open paper qualifies; use in service of a commercial offering is a separate question.

Full text: https://polyformproject.org/licenses/strict/1.0.0


## 1. Clone


In [ ]:
import os, sys, json, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                           capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes — stashing:\n" + dirty)
        !git -C $REPO_DIR stash -u
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
!git -C $REPO_DIR log --oneline -1

## 2. Install


In [ ]:
%pip install -q -e '/content/behaviour-microscope[vllm]'
# vLLM powers the reasoning run. It replays CUDA graphs and batches decode, where the eager
# backend generates one prompt at a time -- 6.2 tok/s on this model, which is where run
# 20260904T075505Z's 4h50m went. Only needed when RUN_THINKING is on; harmless otherwise.
# It comes in through the project's own `vllm` extra rather than a bare `pip install vllm`, so
# vLLM's torch pin and interp-engine's are settled in one resolve instead of two.

# That resolve can still leave Colab's preinstalled torchaudio built against a different CUDA
# than the torch that ends up installed (13.0 vs 12.8 on 2026-09-04). transformers imports
# torchaudio lazily while resolving a model class and rewrites the failure as a missing class,
# so the skew surfaces four cells down as `ModuleNotFoundError: Could not import module
# 'Qwen3_5MoeForCausalLM'` -- the class is present, its import chain is not. Nothing here
# decodes audio, so an unimportable torchaudio is dropped rather than rebuilt.
import subprocess, sys

_probe = subprocess.run([sys.executable, "-c", "import torchaudio"], capture_output=True, text=True)
if _probe.returncode:
    print("torchaudio does not import:", (_probe.stderr.strip().splitlines() or ["?"])[-1])
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
    sys.modules.pop("torchaudio", None)
    print("removed torchaudio -- unused here, and transformers skips it once it is gone")
    if "transformers" in sys.modules:
        print("\nRESTART REQUIRED: transformers is already imported and has cached torchaudio as\n"
              "available. Runtime -> Restart session, then run from cell 1.")
else:
    print("torchaudio imports cleanly")

import importlib, interp_engine
importlib.reload(interp_engine)
print("installed | vLLM available:", interp_engine.vllm_installed())

## 3. Hardware

Weights are 70.2 GB. The 40GB A100 will OOM during load; this cell exits first.


In [ ]:
import torch

WEIGHTS_GB = 70.2
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU."
props = torch.cuda.get_device_properties(0)
total = props.total_memory / 1e9
print(f"{props.name}  |  {total:.1f} GB  |  compute {props.major}.{props.minor}")

headroom = total - WEIGHTS_GB
if headroom < 4:
    raise SystemExit(
        f"This card has {total:.0f} GB; weights are {WEIGHTS_GB} GB. "
        "Use the A100-80GB (High RAM) runtime.")
print(f"Headroom after weights: ~{headroom:.0f} GB")

DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
DEVICE = "cuda"
free_disk = os.statvfs('/content').f_bavail * os.statvfs('/content').f_frsize / 1e9
print(f"dtype: {DTYPE}   device: {DEVICE}   free disk: {free_disk:.0f} GB (need ~{WEIGHTS_GB:.0f})")


## 4. Compatibility

Meta-device skeleton (no weights). Confirms interp-engine can address the required points before a 70 GB download.


In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from interp_engine import EagerModel

MODEL_ID = "thomsonreuters/Thomson-1.0-Small"

cfg = AutoConfig.from_pretrained(MODEL_ID)
print("architecture:", (cfg.architectures or ['?'])[0])

# AutoModelForCausalLM deliberately, not the ForConditionalGeneration class the card names:
# interp-engine's EagerModel loads through AutoModelForCausalLM.from_pretrained, and a skeleton
# built from a different class would have a different module tree, so the points checked below
# would not be the points the run addresses.
try:
    with torch.device("meta"):
        skeleton = AutoModelForCausalLM.from_config(cfg)
except (ImportError, ModuleNotFoundError) as exc:
    raise SystemExit(
        f"{exc}\n\n"
        "transformers reports an import failure *inside* a modeling module as a missing class, so "
        "this is usually a broken optional dependency rather than an unsupported architecture -- "
        "the line above printed the architecture, so the config itself parsed. Re-run the install "
        "cell (its torchaudio guard), restart the runtime, and come back."
    ) from exc

tok = AutoTokenizer.from_pretrained(MODEL_ID)
probe = EagerModel(MODEL_ID, hf_model=skeleton, tokenizer=tok)

print(f"layers: {probe.n_layers}   d_model: {probe.d_model}   "
      f"residual streams: {probe.residual_basis.n_streams}   lens valid: {probe.residual_basis.lens_valid}")
print()
ok = True
for name in ["resid_post", "resid_pre", "mlp_out", "attn_out", "router_logits"]:
    try:
        mod, side = probe.resolve_point(name, 10)
        print(f"  {name:14s} OK   {type(mod).__name__} ({side})")
    except Exception as e:
        ok = False
        print(f"  {name:14s} FAIL {type(e).__name__}: {e}")

assert ok, "Required interp-engine point unresolved — abort before download."
print("resid_post is required for experiments 2–4.")
del probe, skeleton

## 5. Scenarios

Same 30 items, seven arms, and answer key as the other runs.


In [ ]:
from microscope.experiment import RunConfig, run_sweep, compare_runs
from microscope.scenarios import ARMS, load_scenarios
import pandas as pd

scenarios = load_scenarios()
print(len(scenarios), "scenarios")
for arm in ARMS:
    print(f"  {arm.name:20s} {arm.cue or '(no assertion)'}")

## 6. Run

Thomson-1 reads `enable_thinking` and emits `<think>`:

1. Reasoning off — comparable to gemma / Qwen-off; mechanistic sweep available.
2. Reasoning on — behavioural only (answer is not at the final prompt position).

**Reasoning-off is switched off**, because `results/20260904T071909Z` already has it and that
run is unaffected by the parse fix: with reasoning off the answer is read from the first
token's logits, so the reasoning-block parser is never called. Set `RUN_PLAIN = True` to
reproduce it — note it needs `BACKEND_THINKING`'s counterpart, `backend="eager"`, since the
mechanistic sweep cannot run on a CUDA-graph backend.

Reasoning-on is being re-run because `results/20260904T075505Z` is invalid: its completions
carried a closing `</think>` with no opening tag, so the block was never stripped and the
answer letter was read out of the echoed option list — 152 of 210 rows wrong, and a floor
accuracy of 43% that was an artifact. That run also used a 512-token budget and truncated 54%
of completions mid-thought without flagging them. Both fixed; the budget is now 2048.

A reasoning run captures nothing, so it does not need the eager backend's forward hooks —
hence `vllm-generate`, which replays CUDA graphs with no taps. Expect minutes rather than the
previous 4h50m. The preflight cell below spends one prompt confirming the answer parses before
committing to 210.

In [ ]:
MECHANISTIC = True     # False -> experiment 1 only, far faster on a 35B MoE
RUN_PLAIN = False      # reasoning-off: already in results/20260904T071909Z, unaffected by the fix
RUN_THINKING = True    # reasoning-on: re-run, the previous one mis-parsed every answer

# A reasoning run is behavioural-only, so it can use the fast CUDA-graph backend. The plain run
# cannot: capture and patching need the Python forward hooks that only the eager backend runs.
BACKEND_THINKING = "vllm-generate"
BACKEND_PLAIN = "eager"

def thomson(thinking):
    return RunConfig(
        model_id=MODEL_ID, provider="local", dtype=DTYPE,
        backend=BACKEND_THINKING if thinking else BACKEND_PLAIN,
        # vLLM owns its own device placement and rejects a device kwarg; eager needs one.
        extra_load_kwargs={} if thinking else {"device": DEVICE},
        n_candidate_layers=4,
        enable_thinking=thinking,
        mechanistic=MECHANISTIC,   # a reasoning run is behavioural-only regardless
        # arms=("floor", "junior_said", "partner_said", "partner_confirmed", "court"),
    )

configs = [thomson(thinking=t) for t, on in ((False, RUN_PLAIN), (True, RUN_THINKING)) if on]
if not configs:
    raise SystemExit("Nothing to run: set RUN_PLAIN or RUN_THINKING.")
for c in configs:
    print(f"  reasoning {'on ' if c.enable_thinking else 'off'}  backend={c.backend}")

### 6a. Preflight

One prompt through the configured backend, to confirm the answer parses before committing to
210 — the check run `20260904T075505Z` did not have, which is why its mis-parse was only
visible 4h50m later.

It loads the weights, tears them down and reports what was released. That is a second 70 GB
load, so it costs a few minutes; set `PREFLIGHT = False` to skip it. Watch the released figure:
the previous session reported `69.3 GB -> 69.3 GB` between models, and if memory does not come
back here the full run will OOM on load.

In [ ]:
PREFLIGHT = True

if PREFLIGHT:
    from microscope.backends import BackendSpec
    import gc

    _cfg = configs[-1]
    _opts = dict(_cfg.extra_load_kwargs)
    _opts.update(backend=_cfg.backend, enable_thinking=_cfg.enable_thinking, dtype=_cfg.dtype)
    _before = torch.cuda.memory_reserved() / 1e9
    _probe_backend = BackendSpec(kind="local", model_id=MODEL_ID, options=_opts,
                                 max_gen_tokens=_cfg.max_gen_tokens).build()
    try:
        _s = scenarios[0]
        _m = _probe_backend.measure(_s.prompt("floor"))
        print("engine backend :", _probe_backend.handle.backend)
        print("can capture    :", _probe_backend.handle.can_capture)
        print("response mode  :", _probe_backend.response_mode)
        print("gen budget     :", _probe_backend.max_gen_tokens)
        print("parsed letter  :", _m.chosen_letter, " expected:", _s.correct_letter)
        print("parse_ok       :", _m.parse_ok, "| source:", _m.probability_source)
        print()
        print("completion tail:", repr(_m.generated[-200:]))
        assert _m.parse_ok, "No answer parsed -- do not start the full run."
        assert _m.probability_source != "text_truncated", (
            "Truncated mid-reasoning at this budget. Raise max_gen_tokens before running.")
        # Not asserted: one scenario is not a measurement, and a wrong answer here is a result
        # rather than a fault. Only the parse is being checked.
        print("answer correct :", _m.chosen_letter == _s.correct_letter)
        print("\nPreflight OK.")
    finally:
        _probe_backend.shutdown()
        del _probe_backend
        gc.collect()
        torch.cuda.empty_cache()
        _after = torch.cuda.memory_reserved() / 1e9
        print(f"\nGPU reserved: {_before:.1f} GB -> {_after:.1f} GB")
        if _after > _before + 4:
            print("WARNING: memory did not come back. Restart the runtime before the full run.")
else:
    print("Preflight skipped.")

### 6b. Run

In [ ]:
import concurrent.futures
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
    runs = pool.submit(run_sweep, configs).result()
runs

## 7. Results


In [ ]:
for label, path in runs.items():
    s = json.loads((path / "summary.json").read_text())
    q = json.loads((path / "quality_report.json").read_text())
    print(f"=== {label} — quality gate: {q['overall'].upper()} ===")
    for arm, rate in s["behavioural"]["fpar_by_arm"].items():
        acc = s["behavioural"]["accuracy_by_arm"][arm]
        print(f"  {arm:20s} accepts false {rate:5.0%}   accuracy {acc:4.0%}")
    print()

In [ ]:
# Against the other models, if their runs are on this machine.
from pathlib import Path
everything = dict(runs)
for p in sorted(Path("results").glob("*Z")):
    if not (p / "summary.json").exists() or p in runs.values():
        continue
    m = json.loads((p / "manifest.json").read_text())
    everything.setdefault(m["model"], p)

table = compare_runs(everything)
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

## 8. Scope

Forced-choice, n=30, England and Wales. Not a measurement of CoCounsel as a deployed system (retrieval, prompting, and product guardrails are out of band). Planned contrast of interest: `partner_confirmed` vs `court`.

If publishing an unfavourable result that names the product, notify Thomson Reuters and offer a right of reply.


## 9. Export


In [ ]:
import shutil
for label, path in runs.items():
    archive = shutil.make_archive(f"/content/{path.name}", "zip", path)
    print(archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print(f"  download from the file browser instead ({exc})")